# Training visualization — LGD

End-to-end view of every continued-pretraining trial that
`scripts/train_pipeline.py` has produced — per-epoch loss/metric
curves, learning-rate schedule, cross-trial comparisons,
convergence diagnostics, and resource-usage trade-offs.

## Where the data comes from

Two artefacts written by the training pipeline:

* **Per-epoch CSV** — one file per trial under
  `output/training/epochs/<track>/<descriptive_name>.csv` with the
  schema `(epoch, train_loss, lr, metric_name, train_metric,
  test_metric, epoch_time_sec, elapsed_sec)`.
* **Run manifest CSV** — `output/training/manifests/<run_name>_<track>.csv`,
  one row per trial: hyperparameters, status (OK / FAIL), total
  walltime, the path of the final-epoch checkpoint.

All loading and plotting code lives in
`src/utils/training_viz.py`; this notebook contains only function
calls so the notebook stays scannable and the logic stays testable.

## What a 'trial' is

One trial = one `(base_checkpoint × learning_rate × use_lora)` tuple
from `cfg.tunable` in `config/train.yaml`. The full grid is the
Cartesian product, so a four-base / four-LR / {no-LoRA, LoRA}
sweep is 32 trials per track. SLURM array indices map 1-to-1 onto
trial indices.

## Direction of improvement

Higher is better for PD (`roc_auc`); lower is better for LGD
(`rmse`). Every helper in this notebook is direction-aware — `best`
and `sort_values` calls flip sign automatically.

## Sections

1. Setup and what's on disk
2. Per-trial dashboards (one model at a time)
3. Cross-trial loss & metric curves
4. Final-metric comparisons by hyperparameter
5. Time / accuracy trade-off
6. Convergence diagnostics
7. Failures and leaderboard

In [ ]:
%matplotlib inline
import sys, os
from pathlib import Path
REPO = Path(os.getcwd()).resolve()
# pyproject.toml, not a module path: src/visualize/ used to be
# src/utils/ and this loop then walked past the repo root.
while not (REPO / 'pyproject.toml').exists() and REPO.parent != REPO:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

# `display` is an IPython builtin. src/utils/run_notebooks.py executes this file as
# a plain script to rebuild All_Results.md, where it is not — so import it.
from IPython.display import display

import pandas as pd
from src.visualize.figures import FigureSaver
from src.visualize import style
from src.visualize.training_viz import (
    load_run_manifest, load_epoch_history, load_all_epoch_histories,
    training_overview, trial_leaderboard, failed_trials,
    plot_loss_curve, plot_lr_schedule, plot_metric_curves,
    plot_epoch_time, plot_trial_dashboard,
    plot_loss_overlay, plot_metric_overlay, plot_overfitting_diagnostic,
    plot_final_metric_bar, plot_lr_effect, plot_lora_effect,
    plot_metric_heatmap, plot_base_ranking,
    plot_pareto_time_vs_metric, plot_epoch_time_overlay,
    plot_convergence_speed,
    plot_weight_drift, plot_per_dataset_loss,
)
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)
TRACK = 'lgd'    # this notebook is fixed to the LGD track
sink = FigureSaver('1.1. training_visualization_lgd')

# One style for every figure in this project.
style.apply()


## 1 · What's on disk

Manifest tells us which trials were attempted and their outcome;
the per-epoch CSV count tells us how many of those produced any
training history.

In [ ]:
manifest = load_run_manifest(TRACK)
manifest

In [ ]:
overview = training_overview(TRACK)
overview

## 2 · Per-trial dashboard

Pick one trial (any descriptive_name from the leaderboard). The
dashboard gives a 2×2 view of `loss`, `lr`, `train/test metric`,
and `epoch wall-clock` — useful for spotting one slow chunk, an
exploding loss, or a metric that's still climbing at the cliff.

In [ ]:
# Default: the best-scoring trial in the overview. Override with any
# trial_name string to inspect that one instead.
FOCUS_TRIAL = (
    overview.sort_values('best_test_metric', ascending=(TRACK == 'lgd'))['trial_name'].iloc[0]
    if not overview.empty else 'creditpfn_pd_tabpfn-v3-classifier-v3_default_lr5e-05_seed42'
)
FOCUS_TRIAL

In [ ]:
sink.save(plot_trial_dashboard(FOCUS_TRIAL, TRACK), 'trial_dashboard',
          caption="Four panels for the focus trial: training loss, learning-rate schedule, train and held-out monitoring metric, and wall-clock seconds, all against epoch.")

In [ ]:
sink.save(plot_loss_curve(FOCUS_TRIAL, TRACK), 'loss_curve',
          caption="Training loss against epoch for the focus trial, averaged over the optimiser steps of each epoch.")

In [ ]:
sink.save(plot_lr_schedule(FOCUS_TRIAL, TRACK), 'lr_schedule',
          caption="Learning rate against epoch for the focus trial: linear warmup followed by cosine decay to the configured floor.")

In [ ]:
sink.save(plot_metric_curves(FOCUS_TRIAL, TRACK), 'metric_curves',
          caption="Monitoring metric against epoch for the focus trial, on the training datasets and on the held-out datasets, on twin axes. The monitor uses a fixed 2 000-row sample per dataset.")

In [ ]:
sink.save(plot_epoch_time(FOCUS_TRIAL, TRACK), 'epoch_time',
          caption="Wall-clock seconds per epoch for the focus trial.")

## 3 · Cross-trial overlays

Every trial on one axes. Colour = base checkpoint;
solid = no-LoRA, dashed = LoRA. Use these to spot regimes:

* If LoRA trials cluster well below no-LoRA ⇒ LoRA is hurting on
  this track and base.
* If one base sits clearly above the others throughout ⇒ pick that
  base as the headline checkpoint.
* If test_metric falls while train_metric keeps climbing ⇒ overfit
  starting; check the gap plot.

In [ ]:
sink.save(plot_loss_overlay(TRACK), 'loss_overlay',
          caption="Training loss against epoch for every trial in the grid, one line per trial, coloured by base checkpoint and dashed for the adapter arm.")

In [ ]:
sink.save(plot_metric_overlay(TRACK, split='test'), 'metric_overlay',
          caption="Held-out monitoring metric against epoch for every trial, one line per trial, coloured by base checkpoint.")

In [ ]:
sink.save(plot_metric_overlay(TRACK, split='train'), 'metric_overlay',
          caption="Held-out monitoring metric against epoch for every trial, one line per trial, coloured by base checkpoint.")

In [ ]:
sink.save(plot_overfitting_diagnostic(TRACK), 'overfitting_diagnostic',
          caption="Training metric minus held-out metric against epoch, one line per trial. A rising line is the training corpus being fitted at the expense of unseen datasets.")

## 4 · Final-metric comparisons by hyperparameter

**`best_test_metric`** is the best test-set score the trial
achieved during training (max of `test_metric` for PD,
min for LGD).

**`final_test_metric`** is the last-epoch value (i.e. what the
checkpoint actually delivers on disk).

Swap the `metric=` kwarg below to view the latter instead.

In [ ]:
sink.save(plot_final_metric_bar(TRACK, metric='best_test_metric'), 'final_metric_bar',
          caption="Final monitoring metric per trial, sorted, one bar per trial, coloured by base checkpoint.")

In [ ]:
sink.save(plot_metric_heatmap(TRACK, metric='best_test_metric'), 'metric_heatmap',
          caption="Final monitoring metric over base checkpoint against learning rate, one panel per adaptation mode.")

In [ ]:
sink.save(plot_lr_effect(TRACK, metric='best_test_metric'), 'lr_effect',
          caption="Final monitoring metric against learning rate, one line per (base checkpoint, adaptation mode) pair.")

In [ ]:
sink.save(plot_lora_effect(TRACK, metric='best_test_metric'), 'lora_effect',
          caption="Final monitoring metric of the full-finetuning arm against the adapter arm, paired within each (base, learning rate) pair. The diagonal marks equality.")

In [ ]:
sink.save(plot_base_ranking(TRACK, metric='best_test_metric'), 'base_ranking',
          caption="Distribution of the final monitoring metric per base checkpoint, over all learning rates, adaptation modes and query fractions.")

## 5 · Time / accuracy trade-off

The Pareto-frontier-style scatter pairs **total training time** (x,
minutes) with the **final-epoch test metric** (y). A trial that
matches another's metric in less time pareto-dominates it.

The per-epoch bar plot underneath flags trials whose mean
seconds/epoch is unusually high — useful when one dataset's
chunk drags the whole epoch budget.

In [ ]:
sink.save(plot_pareto_time_vs_metric(TRACK, metric='best_test_metric'), 'pareto_time_vs_metric',
          caption="Total training wall-clock against final monitoring metric, one point per trial, coloured by base checkpoint.")

In [ ]:
sink.save(plot_epoch_time_overlay(TRACK), 'epoch_time_overlay',
          caption="Median seconds per epoch per trial, one bar per trial, coloured by base checkpoint.")

## 6 · Convergence diagnostics

Where in the training budget does the best test metric appear?

* Most mass at < 30 % ⇒ early stop is leaving training cycles on
  the floor; consider shorter epochs or a stricter LR schedule.
* Most mass at > 80 % ⇒ trials are still improving when training
  ends; increase `cfg.train.epochs` before re-launching the sweep.

In [ ]:
sink.save(plot_convergence_speed(TRACK), 'convergence_speed',
          caption="Distribution of the epoch at which each trial reached its best held-out monitoring metric.")

## 7 · Failures and leaderboard

`failed_trials()` surfaces any FAIL rows from the manifest with
their error string — most often a `CUDA OOM` from a too-large
`n_finetune_ctx_plus_query_samples`. The leaderboard at the end
is the canonical sorted view used to pick the headline
checkpoint for eval.

In [ ]:
failures = failed_trials(TRACK)
if failures.empty:
    print('No failed trials. ✔')
else:
    display(failures)

In [ ]:
trial_leaderboard(TRACK)

## 8. Did the model actually move?

`drift__<stage>` is $\|w - w_0\|$ over one top-level module of the model, recorded on
every monitored epoch. This is the plot that reframed run-5: a trial whose drift stays
flat near zero has **not trained**, so its "no effect" result says nothing about
continued pretraining. Below it, the per-dataset loss shows *which* tables the model is
learning — a corpus loss that falls while one dataset's loss rises is the model trading
one table off against the others.

In [ ]:
sink.save(plot_weight_drift(TRACK), 'weight_drift',
          caption="Distance of each trial's weights from its base checkpoint against epoch, on a log axis, taking the maximum over the model's top-level stages. One line per trial, coloured by base checkpoint.")


In [ ]:
sink.save(plot_per_dataset_loss(FOCUS_TRIAL, TRACK), 'per_dataset_loss',
          caption="Mean training loss per dataset against epoch for the focus trial, one line per training dataset, ordered in the legend by final loss.")


In [ ]:
lb = trial_leaderboard(TRACK)
print(f'## 1.1 training visualization ({TRACK})')
print(f'trials in manifest: {len(lb)}')
print(lb.to_string() if len(lb) else '  no trials')
print()
print(sink.summary())
